# 평균 높이는 영역과 해상도에 따라 어떻게 달라지는가?

영역별 통계에서 공간 범위와 계산 격자의 역할 확인

**할 일** · 평지·산지 두 영역에서 30 m·300 m 평균 비교

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
dem = ee.Image('USGS/SRTMGL1_003').select('elevation')
regions = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Rectangle([126.85,37.47,126.95,37.53], geodesic=False), {'region':'southwest'}),
    ee.Feature(ee.Geometry.Rectangle([126.95,37.62,127.03,37.69], geodesic=False), {'region':'north'})])
stats30 = dem.reduceRegions(regions, ee.Reducer.mean(), 30)
stats300 = dem.reduceRegions(regions, ee.Reducer.mean(), 300)

In [ ]:
for scale, stats in [(30, stats30), (300, stats300)]:
    rows = stats.getInfo()['features']
    print(scale, 'm:', [r['properties'] for r in rows])

In [ ]:
Map = geemap.Map()
Map.centerObject(regions, 10)
Map.addLayer(dem, {'min':0,'max':800}, 'SRTM')
Map.addLayer(regions, {}, 'Regions')
Map

두 영역의 평균 차이와 해상도 변경에 따른 차이를 구분하시오. `scale=300`을 새로운 300 m 관측자료라고 볼 수 있는지 설명하시오.

**해석** · 재표본화는 새로운 관측을 만들지 않음. 영역 경계·마스크·격자에 따른 통계 차이 가능

## 자료와 해석 범위

- [SRTM](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단